# Étape 10 — essai de traduction terminé

**1 200 mises à jour sur NVIDIA L4**, meilleur checkpoint de validation : **100**. Taille inchangée : **20 860 416 paramètres**. Entrées en français seul ; limite technique512. Les 40 questions connues ne servent pas à la sélection.

| Test réservé | chrF++ parent | Candidat |
|---|---:|---:|
| Phrases nouvelles (12) | 14.456 | 12.813 |
| Phrases documentaires (26) | 11.462 | 12.476 |
| Blocs (7) | 12.697 | 13.975 |
| Textes complets (2) | 8.772 | 15.692 |
| Tours écrits nouveaux (5) | 11.252 | 11.361 |

**Bilan : progrès partiel sur les textes longs, mais sorties répétitives, régression sur les phrases nouvelles et dialogues encore inadéquats.** chrF++ n'est pas une validation du sens. Aucune nouvelle sortie n'est exactement identique à sa référence ; une paraphrase correcte peut aussi différer. Pas de relecture native. Seulement deux textes complets et cinq tours écrits, pas des conversations spontanées enregistrées.

Les sorties des deux textes complets passent de **77/87 à 163/174 tokens** sans perdre de mots du français en entrée, mais restent incomplètes et répétitives sur le fond. Les 14 réponses simples sont conservées ; la régression connue baisse de **35/40 à 26/40**. Le candidat reste expérimental ; la référence publique n'est pas remplacée.

**Tout est sauvegardé dans Drive : poids retenus, checkpoint complet à1 200, optimiseur, états aléatoires, rapports et journal réel.** Le formulaire plus bas charge explicitement **étape10** ; celui de l'ancien notebook étape6 charge un autre modèle.

Exécution dans le calcul L4 existant. L'enregistrement automatique du notebook source était en conflit ; ses modifications distantes n'ont pas été écrasées. Le code et son stdout réel ont été récupérés dans une archive Drive indépendante puis copiés ici, avec origine explicitée dans les métadonnées. Ce notebook n'a pas été entraîné séparément. Les données, mesures complètes et limites figurent dans les rapports.


# MandiaCompact — étape 10 : traduction de français et nouveaux dialogues écrits

**État : entraînement terminé ; bilan réel ci-dessus.** Continuation de notre modèle 20 860 416 paramètres, 1 200 mises à jour sur L4. Pas de nouveau poids préentraîné externe.

**Entrées de traduction : texte français seul**, avec le mode interne de traduction existant ; elles ne contiennent ni anglais ni réponse baoulé dans la source. Le français nouveau est traduit par l'assistant depuis l'anglais publié ; le baoulé est copié des traductions SMOL, sous CC-BY-4.0. Ce ne sont pas des conversations spontanées enregistrées. Pas de relecture native.

239 vues préparées : 108 familles de phrases nouvelles, 7 documents traduits en français phrase par phrase et sous forme de blocs, plus des tours de dialogues écrits. TRAIN : 84 phrases isolées, 51 phrases documentaires, 13 blocs, 4 documents complets et 5 nouveaux tours écrits. Les partitions par famille/document sont disjointes. Les vues d'un même document partagent du contenu au sein d'une partition.

**Textes longs : jusqu'à 485 tokens français d'entrée et 393 tokens baoulé de référence.** La limite technique passe à 512 sans ajouter de paramètres. La référence utilise les poids de l'étape9 avec la même limite pour une comparaison possible ; cela ne garantit pas la compréhension des longues séquences.

Sélection sur validation de traductions de nouvelles familles, de textes longs et de dialogues écrits précédents. **Les 40 questions connues ne servent pas à sélectionner le checkpoint.** Cinq nouveaux tours de deux documents et deux textes complets réservés ne sont évalués qu'après sélection ; ils restent de très petits tests.

**Tous les checkpoints, états d'optimiseur et états aléatoires sont sauvegardés dans Drive.** Le meilleur poids et le dernier état de reprise sont distincts.


In [ ]:
# Mandia stage10: French-only translation and longer documents, Drive required.
from google.colab import drive
import pathlib,sys,hashlib,subprocess,importlib,json
if not pathlib.Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
assert pathlib.Path('/content/drive/MyDrive').exists()
ROOT10=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape10')
# Refresh the already-authorized mount only if new cloud assets are not visible.
if not (ROOT10/'AUDIT-DONNEES.json').exists():drive.mount('/content/drive',force_remount=True)
ASSET_HASHES10={'AUDIT-DONNEES.json': '7de92371f66411f26d5a952308c4a6016be44654d4540f6c478e5b583e496ebf', 'source-smoldoc-en-bci.jsonl': 'fa56efac8e1df34fec3b81793677d4f33a3f92cfc187e92794dfe2cf9d1e6fdd', 'francais-documents.json': '61a3801f6ad55ca811c7ad1f905587cb1e03c9743cded90f054aae79f5babd0d', 'nouveaux-paralleles.jsonl': '820ef82951d2a8702fc39738820a585124d93ed9f192f6e04429fe2a8e31a768', 'mandia_stage10.py': 'e7d1726c78c707a46ac7482bdaad29eb8a63c1fac9bc42de3b503a3e111bffce', 'francais-sentences.tsv': '0a0447d793d6c3de5116cad67d4672669731643a44fa6400fcb5c1fce6671954', 'preparer-donnees.py': '2f62b2b069306baba7911b479d7e740a01149296348ed08461379b16b81aa6a6', 'source-smolsent-en-bci.jsonl': 'f5250fee53b2bd297a4fa7f818514987cbb320e03fbccd28dc7696f9360adfe6'}
FROZEN10={'mandia-compact-s2s-etape4/parallel-prepare.jsonl': 'dc2bfd9ba783c5edec159ec35bfbb66206598520bef31f960da3ad92cc240cbd', 'mandia-compact-s2s-etape4/replies-prepare.jsonl': '29334943cd507b8ffa9385b530b9625aec82c0741aeade3575a30c7df7e2caf1', 'mandia-compact-s2s-etape4/data/waxal-baoule-text.jsonl': '4f7730d32d3324d2cc819f33d7633b2d59e01f81c8e2f38651d0377050ec69d4', 'mandia-compact-s2s-etape5/quotidien-prepare.jsonl': '370d3bb631a4ad4765fbe3b15e0f6019c0470ce03c687f87ac71f6fda6b3048c', 'mandia-compact-s2s-etape6/dialogues-prepare.jsonl': '9375c6a0918a5bf8056cecd42df0c8aff90c3439220bc45bfe10e8cc700a2184', 'mandia-compact-s2s-etape7/teacher-grounded-qa.jsonl': '3cfe3c27ed8e221cad480755f9ee1a0fcee1c67583f58744cda427932e94910b', 'mandia-compact-s2s-etape7/mandia_stage7.py': '6044fb5633f0d0438000f005849f78db20c3b6c5edc2b693530818b53322ea41', 'mandia-compact-s2s-etape8/bible-qa-pilote.jsonl': '868ea39a2961c3b590260b3a7fab51581b82aed7db75726c4ab7369d1ede6a0e', 'mandia-compact-s2s-etape8/wide_20m/model.pt': '6f2bac99307900f96d80a6b9d4980a3b5bc19fc07486beff456979b2d3decbee', 'mandia-compact-s2s-etape9/bible-etendue.jsonl': '36bf8130fa9c785e3e1f4ff931903f747c0bc1c064782e3146ba4200618b426c', 'mandia-compact-s2s-etape9/dialogues-ecrits-ajout.jsonl': '2e7a3f0634c96f0f8c91944db74133ca4ff0d0bc968baa4987f208a82782809a', 'mandia-compact-s2s-etape9/expanded_bible/model.pt': 'f9932b33108013ebd0891295b8820d82cc98176e555d755eed0a83ddbcf4d0b9'}

for name,expected in ASSET_HASHES10.items():assert hashlib.sha256((ROOT10/name).read_bytes()).hexdigest()==expected,'Stage10 source changed: '+name
for name,expected in FROZEN10.items():assert hashlib.sha256((ROOT10.parent/name).read_bytes()).hexdigest()==expected,'Frozen project source changed: '+name
try:import tokenizers,sacrebleu
except ImportError:subprocess.check_call([sys.executable,'-m','pip','install','-q','tokenizers==0.22.1','sacrebleu==2.5.1'])
for rel in ['mandia-compact-s2s-etape6/code','mandia-compact-s2s-etape7','mandia-compact-s2s-etape10']:sys.path.insert(0,str(ROOT10.parent/rel))
import mandia_stage10
importlib.reload(mandia_stage10)
print('MANDIA_ETAPE10_DEBUT: nouvelles familles, texte francais seul, contexte512, checkpoints Drive.',flush=True)
report10=mandia_stage10.main()


In [ ]:
import pathlib,json
ROOT=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape10')
r=json.loads((ROOT/'rapport-etape10.json').read_text())
print('MandiaCompact etape10 — entrainement termine sur',r['experiment']['gpu'])
print('Parametres:',r['parameters'],'; mises a jour:',r['training']['steps'],'; meilleur checkpoint sur validation:',r['training']['best_step'])
print('Les 40 questions connues ne servent pas a la selection.')
for k in r['after']:
 if k.startswith('fresh_'):
  a=r['before'][k]['generation'];b=r['after'][k]['generation']
  print(k,': chrF++',round(a['chrF_pp'],3),'->',round(b['chrF_pp'],3),'; exact',round(a['rows']*a['exact_match']),'->',round(b['rows']*b['exact_match']),'/',b['rows'],'; sans EOS',a['no_eos'],'->',b['no_eos'])
print('Qualite native non validee ; seulement 5 nouveaux tours de 2 documents et 2 textes complets reserves.')
print('Checkpoints, optimiseur et etats aleatoires sauvegardes dans Drive.')


In [ ]:
#@title Tester le checkpoint étape10 — texte français seul ou dialogue
texte_fr = "Demain, nous irons au marché pour acheter du riz. Ensuite, nous rentrerons préparer le repas avec toute la famille." #@param {type:"string"}
mode = "Traduire" #@param ["Traduire", "Répondre"]
contexte_fr = "" #@param {type:"string"}
from google.colab import drive
import pathlib,sys,subprocess,json,hashlib
if not pathlib.Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
runtime=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime')
root=runtime/'mandia-compact-s2s-etape10'
try:import torch,tokenizers
except ImportError:subprocess.check_call([sys.executable,'-m','pip','install','-q','tokenizers==0.22.1'])
import torch
from tokenizers import Tokenizer
sys.path.insert(0,str(runtime/'mandia-compact-s2s-etape6/code'));sys.path.insert(0,str(root))
from mandia_seq2seq import MandiaS2S,S2SConfig
from traduire_texte import predict
receipt=json.loads((root/'SAUVEGARDE-VERIFIEE.json').read_text())
path=root/'translation_dialogue/model.pt'
expected=next(f['sha256'] for f in receipt['files'] if f['path']=='translation_dialogue/model.pt')
assert hashlib.sha256(path.read_bytes()).hexdigest()==expected
p=torch.load(path,map_location='cpu',weights_only=True);tp=runtime/'mandia-compact-s2s-etape4/tokenizer.json'
assert hashlib.sha256(tp.read_bytes()).hexdigest()==p['tokenizer_sha256']
tok=Tokenizer.from_file(str(tp));device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model=MandiaS2S(S2SConfig(**p['config'])).to(device);model.load_state_dict(p['state_dict']);model.eval();del p
result=predict(model,tok,texte_fr,mode='translate' if mode=='Traduire' else 'reply',context=contexte_fr)
print('Checkpoint : étape10 / translation_dialogue / model.pt')
print('Baoulé expérimental :',result['text_bci'])
if not result['all_chunks_ended_eos']:print('Attention : au moins une sortie a atteint sa limite et reste inachevée.')
print('Segments traités :',len(result['chunks']),'; qualité linguistique à vérifier.')
del model
